# Datalog

This notebook demonstrates `IndexedIncrementalDatalogBody` on a small transitive-closure program over three edges. The workload runs twice. In the first run, every fact and the program land at outer tick zero, and a single inner fixpoint produces the closure that the cumulative outer-delta exposes. In contrast, the second run drip-feeds the edges one per outer tick, and a fresh fixpoint runs at every outer tick. The sum of per-tick deltas of the drip-fed run must reproduce the batched closure, and that equality is the invariant of the incremental design.

In [ ]:
from pydbsp.circuit import Circuit
from pydbsp.compute import ComputeCtx
from pydbsp.core import Antichain, dbsp_time
from pydbsp.evaluate import Evaluator
from pydbsp.operator import Input, LiftStreamIntroduction
from pydbsp.progress import Time
from pydbsp.indexed_relational_operators import IndexedIncrementalDatalogBody
from pydbsp.storage import DictStorage
from pydbsp.zset import ZSet, ZSetAddition
from pydbsp import datalog as dlg

program = ZSet({
    (("T", ("?X", "?Y")), ("E", ("?X", "?Y"))): 1,
    (("T", ("?X", "?Z")), ("E", ("?X", "?Y")), ("T", ("?Y", "?Z"))): 1,
})
edges = [("E", (0, 1)), ("E", (1, 2)), ("E", (2, 3))]

## Plumbing

The body operator consumes five inputs and emits a paired diff stream of facts and rewrites. Its five inputs are the EDB, the program, the two-dimensional state for facts, the two-dimensional state for rewrites, and the seed rewrite that the rewrite chain needs at outer tick zero. We wire the body once in the helper below.

On every invocation, the helper pushes the supplied EDB chunk and program chunk at the given outer tick, drives the inner fixpoint via `saturate_inner`, and pushes each iteration's diffs back into the two state inputs. As the inner loop runs, the cumulative facts that have been derived at the current outer tick are accumulated and returned to the caller.

In [ ]:
def build_datalog():
    fg = ZSetAddition[dlg.Fact]()
    e = Evaluator[Time](
        circuit=Circuit[Time](),
        storage=DictStorage(),
        ctx=ComputeCtx(lattice=dbsp_time(2)),
        group=fg,
    )
    edb_1d = Input[ZSet[dlg.Fact]](frontier=Antichain(dbsp_time(1))).connect(e.circuit, ())
    program_1d = Input[ZSet[dlg.Rule]](frontier=Antichain(dbsp_time(1))).connect(e.circuit, ())
    state_facts_2d = Input[ZSet[dlg.Fact]](frontier=Antichain(dbsp_time(2))).connect(e.circuit, ())
    state_rewrites_2d = Input[ZSet[dlg.ProvenanceIndexedRewrite]](frontier=Antichain(dbsp_time(2))).connect(e.circuit, ())
    seed_1d = Input[ZSet[dlg.ProvenanceIndexedRewrite]](frontier=Antichain(dbsp_time(1))).connect(e.circuit, ())
    program_2d = LiftStreamIntroduction[ZSet[dlg.Rule]](group=ZSetAddition[dlg.Rule]()).connect(e.circuit, (program_1d,))
    body = IndexedIncrementalDatalogBody(
        fact_group=fg,
        rule_group=ZSetAddition[dlg.Rule](),
        rewrite_group=ZSetAddition[dlg.ProvenanceIndexedRewrite](),
        signal_group=ZSetAddition[dlg.Signal](),
        ext_dir_group=ZSetAddition[dlg.ExtendedDirection](),
        jorder_group=ZSetAddition[tuple[str, dlg.ColumnReference]](),
        gatekeep_group=ZSetAddition[dlg.IndexedGatekeepEntry](),
        indexed_fact_group=ZSetAddition[dlg.IndexedFact](),
    ).connect(e.circuit, (edb_1d, program_2d, state_facts_2d, state_rewrites_2d, seed_1d))
    return e, edb_1d, program_1d, state_facts_2d, state_rewrites_2d, seed_1d, body, fg

def saturate(e, edb_1d, program_1d, sf, sr, seed_1d, body, fg,
             edb_chunk, program_chunk, outer_tick=0):
    e.push(edb_1d, edb_chunk)
    e.push(program_1d, program_chunk)
    if outer_tick == 0:
        e.push(seed_1d, ZSet({(0, dlg._rewrite_monoid.identity()): 1}))
    min_inner = -1
    for o, k in e.frontiers()[sf].elements:
        if o < outer_tick and k > min_inner:
            min_inner = k
    cumulative = fg.identity()
    for k, (diff_facts, diff_rewrites) in e.saturate_inner(
        body, outer_tick,
        is_empty=lambda p: not p[0].inner and not p[1].inner,
        min_inner=min_inner,
    ):
        cumulative = fg.add(cumulative, diff_facts)
        e.push(sf, diff_facts, t=(outer_tick, k))
        e.push(sr, diff_rewrites, t=(outer_tick, k))
    return cumulative

## Batched evaluation

All three edges and the program land at outer tick zero. A single inner fixpoint then produces the closure that the cumulative outer-delta exposes.

In [3]:
e, edb_1d, program_1d, sf, sr, seed_1d, body, fg = build_datalog()
batched = saturate(
    e, edb_1d, program_1d, sf, sr, seed_1d, body, fg,
    edb_chunk=ZSet({e_: 1 for e_ in edges}),
    program_chunk=program,
)
print("batched T facts:", sorted(k[1] for k in batched.inner if k[0] == "T"))

batched T facts: [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]


## Singleton-fed evaluation

Now the same edges arrive one per outer tick. A fresh fixpoint runs at every outer tick, and the per-tick outer-delta is accumulated as it appears. The sum of those deltas must equal the batched result above. That equality is the invariant we set out to verify.

In [4]:
e, edb_1d, program_1d, sf, sr, seed_1d, body, fg = build_datalog()
total = fg.identity()
for t, fact in enumerate(edges):
    outer_delta = saturate(
        e, edb_1d, program_1d, sf, sr, seed_1d, body, fg,
        edb_chunk=ZSet({fact: 1}),
        program_chunk=program if t == 0 else ZSet({}),
        outer_tick=t,
    )
    total = fg.add(total, outer_delta)

print("singleton T facts:", sorted(k[1] for k in total.inner if k[0] == "T"))

singleton T facts: [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
